# Graph Infrastructure Demo: US-201 & US-202

This notebook demonstrates the graph infrastructure built for the Vartakuni Vihāram TSP solver:

- **US-201**: Graph Builder Module - Building and analyzing metro network graphs
- **US-202**: Data Validation Pipeline - Validating metro network data integrity

## Singapore MRT/LRT Network

The Singapore Mass Rapid Transit (MRT) and Light Rail Transit (LRT) network comprises:
- **214 stations** (181 unique locations with interchange stations counted separately)
- **277 connections** (train lines + walking transfers)
- **15 lines** (8 MRT + 7 LRT lines)

In [1]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports
from src.graph import (
    MetroGraphBuilder,
    build_singapore_metro_graph,
    MetroDataValidator,
    validate_metro_data
)
import networkx as nx
import pandas as pd

# Data directory
data_dir = project_root / 'data' / 'raw'

print("✅ Setup complete")
print(f"📂 Data directory: {data_dir}")

✅ Setup complete
📂 Data directory: /Users/akshayr/Library/CloudStorage/OneDrive-Personal/Labs/vartakuni_vihaaram/data/raw


## Part 1: US-202 - Data Validation Pipeline

Before building graphs, we validate the data to ensure quality and catch errors early.

### 1.1 Run Validation Pipeline

In [2]:
# Run validation on Singapore data
print("Running data validation pipeline...\n")

report = validate_metro_data(data_dir)

# Print full report
report.print_report()

Running data validation pipeline...


DATA VALIDATION REPORT
Timestamp: 2025-11-15 16:15:20

Data Files:
  stations: /Users/akshayr/Library/CloudStorage/OneDrive-Personal/Labs/vartakuni_vihaaram/data/raw/stations.csv
  connections: /Users/akshayr/Library/CloudStorage/OneDrive-Personal/Labs/vartakuni_vihaaram/data/raw/connections.csv
  lines: /Users/akshayr/Library/CloudStorage/OneDrive-Personal/Labs/vartakuni_vihaaram/data/raw/lines.csv

Statistics:
  num_stations: 214
  num_connections: 554
  num_lines: 15
  connection_types: {'train': 398, 'walk_between_stations': 84, 'walk_transfer': 72}

Summary:
  Errors:   0
  Warnings: 0
  Info:     1

Issues Found:

ℹ️ INFOS (1):
  [connectivity] Graph is fully connected
    - diameter: 39

✅ VALIDATION PASSED



### 1.2 Validation Summary

The validation pipeline checks for:
- ✅ File existence
- ✅ Data integrity (required fields)
- ✅ Station ID references
- ✅ Travel time validity
- ✅ Distance validity
- ✅ Duplicate connections
- ✅ Graph connectivity

In [3]:
# Detailed summary
summary = report.get_summary()

print("Validation Summary:")
print("=" * 50)
print(f"Total Issues: {sum(summary.values())}")
print(f"  - Errors:   {summary['error']}")
print(f"  - Warnings: {summary['warning']}")
print(f"  - Info:     {summary['info']}")
print()
print(f"Validation Status: {'✅ PASSED' if not report.has_errors() else '❌ FAILED'}")

# Show statistics
print("\nNetwork Statistics:")
print("=" * 50)
for key, value in report.stats.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

Validation Summary:
Total Issues: 1
  - Errors:   0
  - Warnings: 0
  - Info:     1

Validation Status: ✅ PASSED

Network Statistics:
num_stations: 214
num_connections: 554
num_lines: 15
connection_types:
  train: 398
  walk_between_stations: 84
  walk_transfer: 72


### 1.3 Custom Validation with MetroDataValidator

You can also use the `MetroDataValidator` class directly for more control:

In [4]:
# Create validator instance
validator = MetroDataValidator()

# Run validation
custom_report = validator.validate(
    stations_csv=data_dir / 'stations.csv',
    connections_csv=data_dir / 'connections.csv',
    lines_csv=data_dir / 'lines.csv'
)

# Access specific validation results
print(f"Loaded {len(validator.stations)} stations")
print(f"Loaded {len(validator.connections)} connections")
print(f"Loaded {len(validator.lines)} lines")

# Show sample station data
print("\nSample Stations:")
for i, (station_id, data) in enumerate(list(validator.stations.items())[:5]):
    print(f"  {station_id}: {data['station_name']} (Line {data['line_code']})")

Loaded 214 stations
Loaded 554 connections
Loaded 15 lines

Sample Stations:
  BP1: Choa Chu Kang (Line BP)
  BP10: Fajar (Line BP)
  BP11: Segar (Line BP)
  BP12: Jelapang (Line BP)
  BP13: Senja (Line BP)


## Part 2: US-201 - Graph Builder Module

With validated data, we can now build the network graph for TSP algorithms.

### 2.1 Build Singapore Metro Graph (Quick Method)

In [5]:
# Quick build using convenience function
graph = build_singapore_metro_graph(data_dir)

print("Graph Built Successfully!")
print("=" * 50)
print(f"Stations (nodes):     {graph.number_of_nodes()}")
print(f"Connections (edges):  {graph.number_of_edges()}")
print(f"Graph type:           {type(graph).__name__}")
print(f"Is connected:         {nx.is_connected(graph)}")

✅ Graph is fully connected

📊 Graph Statistics:
   Stations: 214
   Connections: 277
   Lines: 15
   Connection types: {'train': 199, 'walk_transfer': 36, 'walk_between_stations': 42}
   Average degree: 2.59
   Diameter: 39
   Average clustering: 0.053
Graph Built Successfully!
Stations (nodes):     214
Connections (edges):  277
Graph type:           Graph
Is connected:         True


### 2.2 Build Graph with MetroGraphBuilder (Advanced)

In [6]:
# Create builder instance
builder = MetroGraphBuilder()

# Build graph from CSV files
builder.build_graph(
    stations_csv=data_dir / 'stations.csv',
    connections_csv=data_dir / 'connections.csv',
    lines_csv=data_dir / 'lines.csv'
)

print("Graph built with MetroGraphBuilder")
print(f"Stations: {len(builder.stations)}")
print(f"Lines: {len(builder.lines)}")

Graph built with MetroGraphBuilder
Stations: 214
Lines: 15


### 2.3 Graph Statistics and Analysis

In [7]:
# Get comprehensive graph statistics
stats = builder.get_graph_stats()

print("Detailed Graph Statistics:")
print("=" * 50)
for key, value in stats.items():
    print(f"{key}: {value}")

Detailed Graph Statistics:
num_stations: 214
num_connections: 277
is_connected: True
num_lines: 15
connection_types: {'train': 199, 'walk_transfer': 36, 'walk_between_stations': 42}
average_degree: 2.59
diameter: 39
average_clustering: 0.053


### 2.4 Validate Graph Connectivity

In [8]:
# Validate connectivity
is_connected, components = builder.validate_connectivity()

print(f"Graph is connected: {is_connected}")
print(f"Number of components: {len(components)}")

if is_connected:
    print("\n✅ All stations are reachable from any starting point!")
else:
    print("\n⚠️ Graph has disconnected components:")
    for i, component in enumerate(components, 1):
        print(f"  Component {i}: {len(component)} stations")
        print(f"    Stations: {sorted(list(component)[:5])}")

Graph is connected: True
Number of components: 0

✅ All stations are reachable from any starting point!


### 2.5 Shortest Path Analysis

Find the shortest path between two stations using Dijkstra's algorithm.

In [9]:
# Example 1: Jurong East to Changi Airport
start = 'NS1'  # Jurong East
end = 'CG2'    # Changi Airport

path, travel_time = builder.get_shortest_path(start, end)

print(f"Shortest Path: {builder.stations[start]['name']} → {builder.stations[end]['name']}")
print("=" * 70)
print(f"Route: {' → '.join(path)}")
print(f"Number of stations: {len(path)}")
print(f"Total travel time: {travel_time:.2f} minutes")

# Show station names
print("\nDetailed Route:")
for i, station_id in enumerate(path, 1):
    station_name = builder.stations[station_id]['name']
    print(f"{i:2d}. [{station_id:5s}] {station_name}")

Shortest Path: Jurong East → Changi Airport
Route: NS1 → EW24 → EW23 → EW22 → EW21 → EW20 → EW19 → EW18 → EW17 → EW16 → EW15 → EW14 → EW13 → EW12 → EW11 → EW10 → EW9 → EW8 → EW7 → EW6 → EW5 → EW4 → CG → CG1 → CG2
Number of stations: 25
Total travel time: 54.33 minutes

Detailed Route:
 1. [NS1  ] Jurong East
 2. [EW24 ] Jurong East
 3. [EW23 ] Clementi
 4. [EW22 ] Dover
 5. [EW21 ] Buona Vista
 6. [EW20 ] Commonwealth
 7. [EW19 ] Queenstown
 8. [EW18 ] Redhill
 9. [EW17 ] Tiong Bahru
10. [EW16 ] Outram Park
11. [EW15 ] Tanjong Pagar
12. [EW14 ] Raffles Place
13. [EW13 ] City Hall
14. [EW12 ] Bugis
15. [EW11 ] Lavender
16. [EW10 ] Kallang
17. [EW9  ] Aljunied
18. [EW8  ] Paya Lebar
19. [EW7  ] Eunos
20. [EW6  ] Kembangan
21. [EW5  ] Bedok
22. [EW4  ] Tanah Merah
23. [CG   ] Tanah Merah
24. [CG1  ] Expo
25. [CG2  ] Changi Airport


In [10]:
# Example 2: Marina Bay to Sengkang
start = 'NS27'  # Marina Bay
end = 'NE16'    # Sengkang
path, travel_time = builder.get_shortest_path(start, end)

print(f"Shortest Path: {builder.stations[start]['name']} → {builder.stations[end]['name']}")
print("=" * 70)
print(f"Number of stations: {len(path)}")
print(f"Total travel time: {travel_time:.2f} minutes")

print("\nRoute:")
for i, station_id in enumerate(path, 1):
    station_name = builder.stations[station_id]['name']
    print(f"{i:2d}. [{station_id:5s}] {station_name}")

Shortest Path: Marina Bay → Sengkang
Number of stations: 15
Total travel time: 26.69 minutes

Route:
 1. [NS27 ] Marina Bay
 2. [NS26 ] Raffles Place
 3. [NS25 ] City Hall
 4. [NS24 ] Dhoby Ghaut
 5. [NE6  ] Dhoby Ghaut
 6. [NE7  ] Little India
 7. [NE8  ] Farrer Park
 8. [NE9  ] Boon Keng
 9. [NE10 ] Potong Pasir
10. [NE11 ] Woodleigh
11. [NE12 ] Serangoon
12. [NE13 ] Kovan
13. [NE14 ] Hougang
14. [NE15 ] Buangkok
15. [NE16 ] Sengkang


### 2.6 Explore Station Data

Access detailed station and line information.

In [11]:
# Get station information
station_id = 'CC1'  # Dhoby Ghaut
station_info = builder.get_station_info(station_id)

print(f"Station Information: {station_id}")
print("=" * 50)
for key, value in station_info.items():
    print(f"{key}: {value}")

Station Information: CC1
name: Dhoby Ghaut
line_code: CC
latitude: 1.298786
longitude: 103.84502
operational_status: active


In [12]:
# Get line information
line_code = 'TE'  # Thomson-East Coast Line
line_info = builder.get_line_info(line_code)

print(f"Line Information: {line_code}")
print("=" * 50)
for key, value in line_info.items():
    print(f"{key}: {value}")

Line Information: TE
name: Thomson-East Coast Line
color: #9D5B25
type: mrt


### 2.7 Analyze Interchange Stations

Find stations that serve multiple lines.

In [14]:
# Find all interchange stations
from collections import defaultdict

# Group stations by name
stations_by_name = defaultdict(list)
for station_id, data in builder.stations.items():
    station_name = data['name']
    stations_by_name[station_name].append((station_id, data['line_code']))

# Find interchanges (stations with multiple codes)
interchange_stations = {
    name: stations 
    for name, stations in stations_by_name.items() 
    if len(stations) > 1
}

print(f"Found {len(interchange_stations)} interchange stations:\n")
for name, stations in sorted(interchange_stations.items())[:30]:
    lines = ', '.join([f"{code} ({id})" for id, code in sorted(stations)])
    print(f"  {name}: {lines}")

print(f"\n... and {len(interchange_stations) - 10} more interchange stations")

Found 30 interchange stations:

  Bayfront: CE (CE1), DT (DT16)
  Bishan: CC (CC15), NS (NS17)
  Botanic Gardens: CC (CC19), DT (DT9)
  Bugis: DT (DT14), EW (EW12)
  Bukit Panjang: BP (BP6), DT (DT1)
  Buona Vista: CC (CC22), EW (EW21)
  Caldecott: CC (CC17), TE (TE9)
  Chinatown: DT (DT19), NE (NE4)
  Choa Chu Kang: BP (BP1), NS (NS4)
  City Hall: EW (EW13), NS (NS25)
  Dhoby Ghaut: CC (CC1), NE (NE6), NS (NS24)
  Expo: CG (CG1), DT (DT35)
  HarbourFront: CC (CC29), NE (NE1)
  Jurong East: EW (EW24), NS (NS1)
  Little India: DT (DT12), NE (NE7)
  MacPherson: CC (CC10), DT (DT26)
  Marina Bay: CE (CE2), NS (NS27), TE (TE20)
  Newton: DT (DT11), NS (NS21)
  Orchard: NS (NS22), TE (TE14)
  Outram Park: EW (EW16), NE (NE3), TE (TE17)
  Paya Lebar: CC (CC9), EW (EW8)
  Promenade: CC (CC4), DT (DT15)
  Punggol: NE (NE17), PTC (PTC)
  Raffles Place: EW (EW14), NS (NS26)
  Sengkang: NE (NE16), STC (STC)
  Serangoon: CC (CC13), NE (NE12)
  Stevens: DT (DT10), TE (TE11)
  Tampines: DT (DT32), E

### 2.8 Network Degree Distribution

Analyze how many connections each station has.

In [15]:
# Calculate degree for each node
degrees = dict(builder.graph.degree())

# Find stations with highest connectivity
top_connected = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Most Connected Stations:")
print("=" * 70)
for station_id, degree in top_connected:
    station_name = builder.stations[station_id]['name']
    print(f"{station_name:25s} [{station_id:5s}] - {degree} connections")

Top 10 Most Connected Stations:
Bencoolen                 [DT21 ] - 11 connections
Downtown                  [DT17 ] - 8 connections
Telok Ayer                [DT18 ] - 8 connections
Bras Basah                [CC2  ] - 7 connections
Jalan Besar               [DT22 ] - 7 connections
Raffles Place             [EW14 ] - 7 connections
Raffles Place             [NS26 ] - 7 connections
Bugis                     [DT14 ] - 6 connections
Bugis                     [EW12 ] - 6 connections
City Hall                 [EW13 ] - 6 connections


In [16]:
# Degree distribution summary
from collections import Counter

degree_counts = Counter(degrees.values())

print("Degree Distribution:")
print("=" * 40)
for degree in sorted(degree_counts.keys()):
    count = degree_counts[degree]
    bar = '█' * (count // 5)
    print(f"{degree} connections: {count:3d} stations {bar}")

Degree Distribution:
1 connections:  15 stations ███
2 connections: 132 stations ██████████████████████████
3 connections:  38 stations ███████
4 connections:  11 stations ██
5 connections:   4 stations 
6 connections:   7 stations █
7 connections:   4 stations 
8 connections:   2 stations 
11 connections:   1 stations 


### 2.9 Direct Access to NetworkX Graph

The graph is a standard NetworkX graph, so you can use all NetworkX algorithms.

In [17]:
# Access the raw NetworkX graph
G = builder.graph

# Example: Find graph diameter (longest shortest path)
diameter = nx.diameter(G)
print(f"Network diameter: {diameter} stations")
print("(This is the maximum number of stations in any shortest path)")

# Example: Calculate average clustering coefficient
avg_clustering = nx.average_clustering(G)
print(f"\nAverage clustering coefficient: {avg_clustering:.4f}")

# Example: Find center of the network (stations with minimum eccentricity)
center = nx.center(G)
print(f"\nNetwork center stations ({len(center)} stations):")
for station_id in center[:5]:
    station_name = builder.stations[station_id]['name']
    print(f"  {station_name} [{station_id}]")

Network diameter: 39 stations
(This is the maximum number of stations in any shortest path)

Average clustering coefficient: 0.0533

Network center stations (2 stations):
  Newton [DT11]
  Little India [DT12]


## Summary

This notebook demonstrated:

### US-202: Data Validation Pipeline
- ✅ Comprehensive validation of metro network data
- ✅ Detection of errors, warnings, and data quality issues
- ✅ Validation reports with detailed statistics
- ✅ Singapore MRT/LRT data passes all validation checks

### US-201: Graph Builder Module
- ✅ Building NetworkX graphs from CSV data
- ✅ Graph connectivity validation
- ✅ Shortest path finding (Dijkstra's algorithm)
- ✅ Station and line information access
- ✅ Network analysis and statistics
- ✅ Fully connected graph with 214 stations and 277 edges

### Next Steps
The validated, fully-connected graph is now ready for TSP algorithm implementation in **Epic 3: TSP Solver Development**!

---

*Vartakuni Vihāram (వర్తకుని విహారం) - A Seller's Journey*